<a href="https://colab.research.google.com/github/adishup/gen-ai-lab/blob/main/experiment%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch

In [ ]:
# ============================================================
# EXPERIMENT 2
# SENTIMENT ANALYSIS AND DOCUMENT CLASSIFICATION
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ============================================================
# PART A — SENTIMENT ANALYSIS
# ============================================================

print("=" * 60)
print("PART A - SENTIMENT ANALYSIS")
print("=" * 60)


# ------------------------------------------------------------
# 1. Select Device
# ------------------------------------------------------------

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("GPU:", torch.cuda.get_device_name(0))

else:

    device = torch.device("cpu")

    print("Using CPU")


# ------------------------------------------------------------
# 2. Load Sentiment Model
# ------------------------------------------------------------

sentiment_model_name = (
    "distilbert-base-uncased-finetuned-sst-2-english"
)

print("\nLoading sentiment model...")

sentiment_tokenizer = AutoTokenizer.from_pretrained(
    sentiment_model_name
)

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    sentiment_model_name
)

sentiment_model = sentiment_model.to(device)

sentiment_model.eval()

print("Sentiment model loaded successfully.")


# ------------------------------------------------------------
# 3. Input Text
# ------------------------------------------------------------

text = (
    "The Generative AI workshop was extremely "
    "informative and useful."
)


# ------------------------------------------------------------
# 4. Tokenize Text
# ------------------------------------------------------------

sentiment_inputs = sentiment_tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

sentiment_inputs = {
    key: value.to(device)
    for key, value in sentiment_inputs.items()
}


# ------------------------------------------------------------
# 5. Predict Sentiment
# ------------------------------------------------------------

with torch.no_grad():

    sentiment_output = sentiment_model(
        **sentiment_inputs
    )


# ------------------------------------------------------------
# 6. Calculate Probability
# ------------------------------------------------------------

probabilities = torch.softmax(
    sentiment_output.logits,
    dim=-1
)

predicted_class = torch.argmax(
    probabilities,
    dim=-1
).item()

confidence = probabilities[
    0,
    predicted_class
].item()


# ------------------------------------------------------------
# 7. Convert Label
# ------------------------------------------------------------

sentiment_labels = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

sentiment = sentiment_labels[
    predicted_class
]


# ------------------------------------------------------------
# 8. Display Sentiment
# ------------------------------------------------------------

print("\nInput:")
print(text)

print("\nSentiment:")
print(sentiment)

print("\nConfidence Score:")
print(round(confidence, 3))


# ============================================================
# PART B — DOCUMENT CLASSIFICATION
# ============================================================

print("\n")
print("=" * 60)
print("PART B - DOCUMENT CLASSIFICATION")
print("=" * 60)


# ------------------------------------------------------------
# 9. Load Zero-Shot Classification Model
# ------------------------------------------------------------

classification_model_name = (
    "facebook/bart-large-mnli"
)

print("\nLoading document classification model...")

classification_tokenizer = AutoTokenizer.from_pretrained(
    classification_model_name
)

classification_model = AutoModelForSequenceClassification.from_pretrained(
    classification_model_name
)

classification_model = classification_model.to(device)

classification_model.eval()

print("Classification model loaded successfully.")


# ------------------------------------------------------------
# 10. Input Document
# ------------------------------------------------------------

document = """
Artificial Intelligence and Machine Learning are transforming
industries through automation and intelligent decision-making.
"""


# ------------------------------------------------------------
# 11. Candidate Labels
# ------------------------------------------------------------

labels = [
    "Technology",
    "Sports",
    "Politics",
    "Entertainment"
]


# ------------------------------------------------------------
# 12. Calculate Scores for Each Label
# ------------------------------------------------------------

scores = []

for label in labels:

    hypothesis = (
        "This document is about " + label + "."
    )

    inputs = classification_tokenizer(
        document,
        hypothesis,
        return_tensors="pt",
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output = classification_model(
            **inputs
        )

    # BART-MNLI labels:
    # contradiction, neutral, entailment

    logits = output.logits[0]

    entailment_score = torch.softmax(
        logits,
        dim=-1
    )[2].item()

    scores.append(entailment_score)


# ------------------------------------------------------------
# 13. Find Best Category
# ------------------------------------------------------------

best_index = max(
    range(len(scores)),
    key=lambda i: scores[i]
)

predicted_category = labels[
    best_index
]

confidence = scores[
    best_index
]


# ------------------------------------------------------------
# 14. Display Classification
# ------------------------------------------------------------

print("\nDocument:")
print(document)

print("\nPredicted Category:")
print(predicted_category)

print("\nConfidence Score:")
print(round(confidence, 3))


# ============================================================
# 15. COMPLETION
# ============================================================

print("\n")
print("=" * 60)
print("EXPERIMENT 2 COMPLETED SUCCESSFULLY")
print("=" * 60)

PART A - SENTIMENT ANALYSIS
Using CPU

Loading sentiment model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Classification model loaded successfully.

Document:

Artificial Intelligence and Machine Learning are transforming
industries through automation and intelligent decision-making.


Predicted Category:
Technology

Confidence Score:
0.401


EXPERIMENT 2 COMPLETED SUCCESSFULLY
